In [ ]:
import torch
from diffusers import QwenImage21Pipeline
from PIL import Image

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' 
print(device)

# Text to Image

In [ ]:
pipe = QwenImage21Pipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1", torch_dtype=torch.bfloat16
).to(device)

In [ ]:
image = pipe(
    prompt="A neon shop sign that reads \"Weights & Biases\", rainy night, reflections on wet pavement",
    num_inference_steps=40,
    generator=torch.Generator(device).manual_seed(42),
).images[0]

image.save("images/t2i_example.png")

# Single Image Editing

In [ ]:
input_image = Image.open("images/t2i_example.png")

image = pipe(
    prompt="Change the colour scheme of the neon sign to be black and yellow and correct the spelling of \"biasses\" to \"biases\"",
    image=input_image,
    num_inference_steps=40,
    generator=torch.Generator(device).manual_seed(42),
).images[0]

image.save("images/single_image_edited.png")

# Multiple Reference Images


In [ ]:
images = [Image.open(f"images/group_photo.png"), Image.open(f"images/wolfram_photo.png")]

result = pipe(
    prompt="Add wolfram_photo into the centre of group_photo, naturally between two of the people in the photo, matching the lighting and perspective of the first image. Don't remove any of the persons in the existing group photo. make the group wider to accomodate the person in photo 2.",
    image=images,
    num_inference_steps=60,
    generator=torch.Generator(device).manual_seed(42),
).images[0]

result.save("images/multi_ref_example.png")

  0%|          | 0/60 [00:00<?, ?it/s]

# Local Region Editing
Two ways to edit a region: the `region_edit` widget (UI), or `edit_with_mask` (programmatic).
Both use the same mask-based core — the widget's **Save mask** button writes a mask file that `edit_with_mask` can consume.

In [ ]:
import os, torch
from PIL import Image, ImageDraw, ImageFilter

def ellipse_mask(size, x, y, w, h):
    """Binary mask (white = region to edit); x, y, w, h are fractions of the image size."""
    m = Image.new("L", size, 0)
    cx, cy, ex, ey = x * size[0], y * size[1], w * size[0] / 2, h * size[1] / 2
    ImageDraw.Draw(m).ellipse([cx - ex, cy - ey, cx + ex, cy + ey], fill=255)
    return m

def marked_from_mask(img, mask, line=None):
    """Copy of img with the mask boundary drawn in red — the annotation the model reads."""
    line = line or max(5, img.width // 300)
    edge = (mask.convert("L").resize(img.size)
                .filter(ImageFilter.FIND_EDGES).filter(ImageFilter.MaxFilter(line | 1))
                .point(lambda p: 255 if p else 0))
    out = img.convert("RGB").copy()
    out.paste((255, 0, 0), (0, 0), edge)
    return out

def edit_with_mask(pipe, img, mask, prompt, steps=40, seed=42, **kw):
    """Programmatic local edit from any white-on-black mask."""
    return pipe(prompt=prompt, image=marked_from_mask(img, mask),
                num_inference_steps=steps,
                generator=torch.Generator("cuda").manual_seed(seed), **kw).images[0]

In [ ]:
import io, os, base64, anywidget, traitlets, ipywidgets as ui
from PIL import ImageChops

class PencilCanvas(anywidget.AnyWidget):
    _esm = """
    function render({ model, el }) {
      const img = new Image(), canvas = document.createElement("canvas");
      const ctx = canvas.getContext("2d");
      let drawing = false, stroke = null;
      canvas.style.cursor = "crosshair"; canvas.style.touchAction = "none";
      img.onload = () => {
        canvas.width = model.get("width");
        canvas.height = Math.round(img.height * model.get("width") / img.width);
        redraw();
      };
      img.src = "data:image/png;base64," + model.get("image_b64");
      function redraw() {
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
        ctx.strokeStyle = "rgba(255,0,0,.6)"; ctx.lineCap = ctx.lineJoin = "round";
        for (const [w, pts] of model.get("strokes")) {
          ctx.lineWidth = w; ctx.beginPath();
          pts.forEach(([x, y], i) => i ? ctx.lineTo(x, y) : ctx.moveTo(x, y));
          ctx.stroke();
        }
      }
      const pos = e => { const r = canvas.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; };
      canvas.addEventListener("pointerdown", e => { drawing = true; stroke = [model.get("brush"), [pos(e)]]; });
      canvas.addEventListener("pointermove", e => {
        if (!drawing) return;
        stroke[1].push(pos(e));
        const p = stroke[1];
        ctx.strokeStyle = "rgba(255,0,0,.6)"; ctx.lineCap = ctx.lineJoin = "round";
        ctx.lineWidth = stroke[0]; ctx.beginPath();
        ctx.moveTo(...p[p.length - 2]); ctx.lineTo(...p[p.length - 1]); ctx.stroke();
      });
      const up = () => {
        if (!drawing) return;
        drawing = false;
        model.set("strokes", [...model.get("strokes"), stroke]); model.save_changes();
      };
      canvas.addEventListener("pointerup", up);
      canvas.addEventListener("pointerleave", up);
      model.on("change:strokes", redraw);
      el.appendChild(canvas);
    }
    export default { render };
    """
    image_b64 = traitlets.Unicode("").tag(sync=True)
    strokes   = traitlets.List([]).tag(sync=True)
    brush     = traitlets.Int(14).tag(sync=True)
    width     = traitlets.Int(768).tag(sync=True)

def region_edit(pipe, img, display_width=768):
    img = img.convert("RGB")
    buf = io.BytesIO()
    img.resize((display_width, round(img.height * display_width / img.width))).save(buf, "PNG")
    canvas = PencilCanvas(image_b64=base64.b64encode(buf.getvalue()).decode(), width=display_width)

    def mask(size):
        sx = size[0] / display_width
        m = Image.new("L", size, 0); d = ImageDraw.Draw(m)
        for wd, pts in canvas.strokes:
            scaled = [(px * sx, py * sx) for px, py in pts]
            lw = max(2, round(wd * sx))
            if len(scaled) > 1: d.line(scaled, fill=255, width=lw, joint="curve")
            for px, py in (scaled[0], scaled[-1]):
                d.ellipse([px - lw / 2, py - lw / 2, px + lw / 2, py + lw / 2], fill=255)
        if fill.value:
            outside = m.copy(); ImageDraw.floodfill(outside, (0, 0), 128)
            m = ImageChops.lighter(m, outside.point(lambda p: 255 if p == 0 else 0))
        return m

    brush     = ui.IntSlider(14, 2, 60, description="brush")
    fill      = ui.Checkbox(True, description="fill enclosed")
    clear     = ui.Button(description="Clear")
    mask_path = ui.Text(value="masks/region_mask.png", description="mask file")
    save      = ui.Button(description="Save mask")
    prompt    = ui.Text(value="In the red marked region, ", layout=ui.Layout(width="60%"))
    run       = ui.Button(description="Edit", button_style="primary")
    out       = ui.Output()

    brush.observe(lambda ch: setattr(canvas, "brush", ch["new"]), "value")
    def on_clear(_): canvas.strokes = []
    def on_save(_):
        os.makedirs(os.path.dirname(mask_path.value) or ".", exist_ok=True)
        mask(img.size).save(mask_path.value)
        with out: print("mask saved to", mask_path.value)
    def on_run(_):
        out.clear_output()
        with out: display(edit_with_mask(pipe, img, mask(img.size), prompt.value))
    clear.on_click(on_clear); save.on_click(on_save); run.on_click(on_run)
    display(ui.VBox([canvas, ui.HBox([brush, fill, clear]),
                     ui.HBox([mask_path, save]), ui.HBox([prompt, run]), out]))

In [ ]:
region_edit(pipe, Image.open("images/group_photo.png"))

In [ ]:
# Programmatic editing from a saved mask (no UI)

img  = Image.open("images/group_photo.png")
mask = Image.open("masks/region_mask.png")  # from the widget's Save mask button — or build one: ellipse_mask(img.size, .5, .5, .3, .2)

In [ ]:
edit_with_mask(pipe, img, mask, "In the red circled region, give the person a green baseball cap")

# Region Editing + Multiple References
Combines the two modes: the base image carries the red region annotation (from a mask),
and reference images supply the content. The model reads image 1 as the edit target and
the rest as references — so the prompt should say what to take from which image, e.g.
*"In the red marked region of the first image, insert the person from the second image"*.

In [ ]:
def edit_region_with_refs(pipe, img, mask, refs, prompt, steps=40, seed=42, **kw):
    """Local region edit guided by reference images.

    img:  base image to edit; mask: white-on-black region mask (from the widget or ellipse_mask)
    refs: a PIL image or list of them (the pipeline supports up to 10 images total)
    """
    refs = refs if isinstance(refs, (list, tuple)) else [refs]
    
    return pipe(prompt=prompt,
                image=[marked_from_mask(img, mask)] + [r.convert("RGB") for r in refs],
                num_inference_steps=steps,
                generator=torch.Generator("cuda").manual_seed(seed), **kw).images[0]

In [ ]:
# Example: mark a region in the group photo, fill it using the reference photo
base = Image.open("images/group_photo.png")
ref  = Image.open("images/wolfram_photo.png")
mask = Image.open("masks/region_mask.png")  # from the widget\'s Save mask button, or: ellipse_mask(base.size, .5, .5, .3, .4)

edited_image = edit_region_with_refs(
    pipe, base, mask, ref,
    "In the red marked region of the first image, insert the person from the second image, "
    "matching the lighting and perspective of the first image",
    seed=12,
)

In [ ]:
edited_image